# Water Quality Prediction & Evolutionary Optimisation
**Date:** April 2025  

‼️Please clone the project from GitHub, or update the following paths:
1. File Data path currently set as "../data/e1_nutrients.csv". 
2. If you want the plot images saved, change all plot savefig paths. Currently, all are commented by default.
 
Github link:https://github.com/mzst-26/water-quality-ml

This notebook covers two parts:
- **Part 1:** Predicting nitrite levels in water samples using machine learning regression models (Linear Regression, Random Forest, Neural Network)
- **Part 2:** Optimising the McCormick function using a Hill Climber and Evolutionary Algorithm

---

## Table of Contents
1. [Libraries & Setup](#1-libraries--setup)
2. [Part 1 – Machine Learning](#1)
   - [2.1 Load Data](#1)
   - [2.2 Exploratory Data Analysis](#2.2)
   - [2.3 Data Preparation](#2.3)

<a id="1-libraries--setup"></a>
---
## 1. Libraries & Setup

In [ ]:
# core libraries
import numpy as np
import pandas as pd

#for visualization
import matplotlib.pyplot as plt
import seaborn as sns

#for checking multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

#image visualisation
from IPython.display import Image, display

#Regression models
import sklearn.linear_model as linearRegression
import sklearn.ensemble as RandomForestRegressor
import sklearn.neural_network as MLPRegressor

#Metrices 
from sklearn.metrics import mean_squared_error, r2_score

# Train-test split
from sklearn.model_selection import train_test_split

# Plot styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All libraries loaded successfully.')

<a id="1"></a>
---
# Part 1 – Machine Learning

### 2.1 Load Data

In [ ]:
#load the data using pandas
data_file = pd.read_csv("../data/e1_nutrients.csv")

Used head(), info() and describe() to explore the data and understand the charactiristic of the data.

In [ ]:
data_file.head()        # first 5 rows
data_file.info()        # column names, data types, missing values
data_file.describe()    # min, max, mean, std for every column

<a id="2.2"></a>
---
### 2.2 Exploratory Data Analysis

In [ ]:
# Visualise feature distributions using KDE plots
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
# Flatten the axes array for easy iteration
for ax, col in zip(axes.flatten(), data_file.columns):
    # Create a KDE plot for each feature
    sns.kdeplot(data_file[col], ax=ax, fill=True, color='steelblue')
    ax.set_title(col)

# Adjust layout and show the plot
plt.suptitle('Feature Distributions (KDE)', fontsize=16)
# plt.savefig('../Analytics/kde.png', dpi=150)
plt.tight_layout()
plt.show()

### KDE Distribution Observation
>Previously, I chose a histogram plot to visualise the data, which, compared to KDE, was missing some critical information.
#### The KDE reveals:
1. DEPTH The KDE shows 6 distinct peaks, one for each depth value. This actually visually proves it's discrete rather than continuous, far more clearly than the histogram did.
2. NITRATE+NITRITE has a clear bimodal distribution (two humps — one near 0 and a second around 5-6). This is a significant finding. It suggests two distinct water conditions exist in the dataset, possibly seasonal or driven by depth.
3. SILICATE also shows a bimodal shape with peaks around 0.5 and 2. Again suggests two different water states.
4. PHOSPHATE similarly shows a small secondary peak around 1.5 after the main peak near 0.2.

>The KDE plots reveal that Depth takes exactly 6 discrete values (0, 10, 20, 30, 
40 and 60 metres), visible as 6 distinct peaks, confirming a controlled repeated 
sampling design. NITRITE and AMMONIA are heavily right-skewed with a sharp peak 
near zero and a long tail, indicating most samples have low concentrations, with 
Sometimes, extreme readings which would require outlier handling after being confirmed by other visualisation methods. More interestingly, 
NITRATE+NITRITE, SILICATE and PHOSPHATE all display bimodal distributions, two 
distinct peaks. This suggests the dataset captures two different water conditions, 
possibly caused by the seasonal variation or possible depth-related nutrient stratification. 
These patterns suggest the relationships in this data are non-linear and complex.


In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(data_file.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix')
plt.tight_layout()
# plt.savefig('../Analytics/correlation_heatmap.png', dpi=150)
plt.show()

## Correlation Heatmap Observation
What does the heatmap tell me?
1. Looking at the NITRATE+NITRITE and PHOSPHATE numbers(0.80), we can conclude that their relationship is the strongest. When one side is high, we can assume the other side could also be high.
2. NITRATE+NITRITE and SILICATE (0.69) have a strong relationship, and again, when one increases, you can see the increase on the other as well.
3. SILICATE and PHOSPHATE (0.61) Moderate positive relationship, same pattern.
4. AMMONIA and NITRATE+NITRITE (-0.22) Slight negative relationship, slightly unusual.

### Most important
### NITRITE and everything else (0.03 to 0.31)
Our most important nutrient, which is the target variable (for this prediction model), has a low relationship with everything else.

### What does that mean?
>This means predicting nitrite is genuinely difficult, and linear relationships alone won't capture it well. The weak correlations between NITRITE and all features suggest that nitrite behaves somewhat independently from the other nutrients, which means Linear Regression will likely struggle compared to Random Forest or Neural Network, which can capture non-linear relationships. 
#### *** These are predictions I made before even creating the model, solely based on the correlation heatmap. *** #### 
 


In [ ]:
# Box plots to identify outliers
data_file.plot(kind='box', subplots=True, layout=(3, 4), figsize=(16, 10), sharex=False, sharey=False)
plt.suptitle('Box Plots – Outlier Detection', fontsize=16)
plt.tight_layout()
# plt.savefig('../Analytics/boxplots.png', dpi=150)
plt.show()

## Box Plot Observation
What does it tell me?
1. DEPTH No outliers, clean even spread across 0-60. Confirms what we already knew: controlled sampling.
2. NITRITE Massive amount of outliers shown as black dots above the whisker. The box itself is tiny and squashed near 0, meaning that 50% of all the values are extremely low, but there are hundreds of readings that are unusually high. This confirms the heavy right skew and tells me outlier handling is definitely needed here.
3. NITRATE+NITRITE Large spread in the box, no extreme outliers shown. More variation in the normal range than NITRITE.
4. AMMONIA shows simmilar behaviour as NITRITE.The box is small and there is a huge dense cluster of outliers above.That means there are hundereds of AMMONIA readings that go well above the normal range.
5. SILICATE Only a handful of outliers (the small circles), and the box has a reasonable spread. Much cleaner than NITRITE or AMMONIA. However the density of the outliers in the 5.8 area is unclear. 
6. PHOSPHATE A few scattered outliers above the whisker but nothing as extreme as NITRITE or AMMONIA. Relatively clean however again the density of the outliers in the narow section of approximitly 1.3 is not very clear. 

### What does that mean for me?

>NITRITE and AMMONIA have the most severe outlier problems, hundreds of data points sitting well above the normal range. These need to be handled in the data preparation step with the IQR method before training the models, otherwise those extreme values will disproportionately influence how the models learn and they skew the results.




In [ ]:
# Pairwise relationships coloured by depth
data_file['Depth_Category'] = data_file['Depth'].astype(str) + 'm'
# Create pairplot with seaborn
sns.pairplot(data_file, 
             hue='Depth_Category', # colour by depth category
             vars=['NITRITE', 'NITRATE+NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE'], # only plot these features
             diag_kind='kde', # use KDE for diagonal plots
             plot_kws={'alpha': 0.3}) # add some transparency to the scatter points to better visualise density and reduce overplotting
plt.suptitle('Pairwise Relationships Coloured by Depth', y=1.02) # adjust title position
# plt.savefig('../Analytics/pairplot.png', dpi=150)
plt.show()

# Drop the helper column after plotting
data_file.drop(columns=['Depth_Category'], inplace=True) 

## DIAGONAL (KDE distributions by depth) and OFF-DIAGONAL SCATTER PLOTS
After some research and viewing different tradeoffs, I decided to visualise the data for the last time using this method to confirm some of the guesses we made with the heatmap and the box plots, and we gathered some important data.
For context, the upper diagonal (triangular) and lower diagonal are exact same but with axes swapped. What we are looking for is only learner patterns with low cluster or reliable upward/downward patterns. 

What is not considered reliable: 
>Fan-shaped, Vertical or Horizontal flat cluster(low change, useless relationship), curved patterns (non-linear such as Nitrate vs Silicate) and random clouds.
### Diagonal KDE findings:

1. SILICATE shows the strongest depth separation, and concentration varies most by depth
2. AMMONIA is nearly identical across all depths. Surface-driven process.
3. NITRATE+NITRITE and PHOSPHATE show partial depth separation; deeper samples trend slightly higher.
4. NITRITE overlaps completely across all depths, meaning depth does not drive nitrite levels

### Scatter plot findings:

5. Depth colour never separates any panel, and nutrients vary independently of depth throughout.
6. NITRATE+NITRITE vs PHOSPHATE shows the clearest upward trend, confirms 0.80 correlation on the heatmap, and the same environmental driver
7. NITRATE+NITRITE vs SILICATE also shows a clear upward trend, confirms a 0.69 correlation on the heatmap.
8. NITRITE forms vertical clusters in every panel, with no relationship to any other nutrient, confirming correlation heatmap findings.
9. AMMONIA vs SILICATE and AMMONIA vs PHOSPHATE show near-zero relationships. Ammonia behaves independently.

The pairwise plot reveals much more critical data. Describing them all would indeed result in some long, boring paragraphs. So here are the two most important findings:

1. The depth colour never cleanly separates any scatter plot; the 6 depth levels are randomly mixed throughout every panel, proving that the depth is not the hidden driver behind the nutrients' relationship and that nutrients actually vary independently of depth.
2. Second, NITRITE behaves fundamentally differently from all other variables; it forms dense vertical clusters in every scatter plot it appears in, showing virtually no relationship with any other nutrient. Every other variable shows at least moderate relationships with each other, particularly NITRATE+NITRITE and PHOSPHATE, which show a clear upward trend confirming their 0.80 correlation.




## EDA Summary

>These findings confirm that nitrite is driven by entirely different environmental processes, explaining why predicting it will be genuinely difficult and providing strong evidence that non-linear models will be required.


<a id="2.3"></a>
---
## 2.3 Data Preparation

### How are we going to prepare the data?
1. Check what we are working with.
2. Handle missing values first
3. Handle Outliers
4. Encode DEPTH as Categorical
5. Check Multicollinearity
6. Split Before Scaling
7. Scale Features
8. Final Check, verify Everything Looks Right
   ### Yes, proper and senior level.

### Data Preparation: 1. Check what we are working with.
Before touching anything I would like to understand the scale of the problem. 

In [ ]:
original_len = len(data_file)
print(f"Total rows: {original_len}") # total number of rows
print(f"Total columns: {data_file.shape[1]}") # total number of columns 
print("\nMissing values:")
print(data_file.isnull().sum()) # check if any of the rows have missing values 
print("\nData types:")
print(data_file.dtypes) #understand the data types
print("\nBasic statistics:") 
print(data_file.describe()) #repeated step just to review the data again before preparation

#### Overview
The dataset is large enough to cap rather than drop. The data is overall clean, and there are no unexpected types nor values. Also, there is no new information since we have already analysed the numbers before.

#### Since there are no missing values, we can go straight to step 3.

### Data Preparation: 3. Handle Outliers
Since we are dealing with the environment and those extreme readings could be environmental pollution, I have decided not to drop the outliers to minimise losing critical data. 
>In fact, the outliers themselves may be the real normal data and the low reading could be decrese of minirals of water by seasonal or occasional events such as Algal blooms. Another reason for not dropping is that if we don't train our models with some outliers, we are basically claiming that those situations do not exist. When the model encounters a high nutrient in the future, it will predict badly.
>Capping keeps the information that something extreme happened while limiting how much it distorts the model.



In [ ]:
# Check how extreme the outliers are before deciding
for col in data_file.columns:
    # Calculate the IQR for the column IQR stands for Interquartile Range, which is a measure of statistical dispersion and is used to identify outliers in a dataset. 
    # It is calculated as the difference between the third quartile (Q3) and the first quartile (Q1) of the data. 
    # The IQR helps to determine the range within which most of the data points lie, 
    # and any data points that fall outside of this range are considered outliers.
    Q1 = data_file[col].quantile(0.25)
    Q3 = data_file[col].quantile(0.75)
    IQR = Q3 - Q1 
    # Define outlier thresholds lower in this case means  any value that is less than 1.5 times the IQR below the first quartile which are considered outliers.
    lower = Q1 - 1.5 * IQR 
    upper = Q3 + 1.5 * IQR 
    # Outliers are defined as any value that is less than the lower threshold or greater than the upper threshbold. 
    outliers = data_file[(data_file[col] < lower) | (data_file[col] > upper)]
    # Print the number and percentage of outliers for each column
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(data_file)*100:.1f}%)")

### Output evaluation
The output confirms why capping is the right choice here: 
>As previously mentioned, visualising data helps with understanding the overall patterns and characteristics of the data, but we might miss some information, such as density, when viewing overlapping data.
1. SILICATE: 0.4% (11 rows) and PHOSPHATE: 0.2% (6 rows) show low density of outliers out of threshold, confirming plot boxes were not visually misleading.
2. AMMONIA: 12.5% (311 rows), the worst offender. If we were to drop the outliers, we would have lost 12% of the real environmental data.
3. NITRATE+NITRITE: 0% as expected, confirming the plot box.
4. NITRITE: 8.4% (209 rows) significant but not catastrophic, capping is correct.
5. DEPTH: 0% — confirmed, no outliers.

#### Summary
> This output also validates the markdown reasoning perfectly. AMMONIA at 12.5% proves that dropping would have been wrong; losing over 300 real pollution event readings would have made the model blind to extreme conditions.

### Data Preparation: 3.2 Cap the outliers

In [ ]:
# Cap outliers instead of dropping
cols_to_cap = ['NITRITE', 'AMMONIA', 'SILICATE', 'PHOSPHATE', 'NITRATE+NITRITE']
for col in cols_to_cap:
    # Calculate the IQR for the specific columns column
    Q1 = data_file[col].quantile(0.25)
    Q3 = data_file[col].quantile(0.75)
    IQR = Q3 - Q1
    # Get the lower and upper thresholds for outliers
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Cap the outliers by replacing values outside the thresholds with the threshold values
    data_file[col] = data_file[col].clip(lower, upper)
    
# print the confirmation that all outliers have been capped and the number of rows preserved
print(f"All outliers capped. Rows preserved: {len(data_file)}")

#### Code evaluation
>NITRATE+NITRITE was included in the capping loop as a precaution despite showing zero outliers in the initial check, ensuring consistency across the pipeline if the dataset is updated or thresholds are adjusted in future.

#### The next step is to prove that the capping worked

In [ ]:
# Verify capping worked
# The same method that was used to get the outliers is now pasted here, only for a linear experience. 
for col in cols_to_cap:
    Q1 = data_file[col].quantile(0.25)
    Q3 = data_file[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data_file[(data_file[col] < lower) | (data_file[col] > upper)]
    
    print(f"{col}: {len(outliers)} outliers remaining")


### Data Preparation: 3.3 Changes are easier to comprehend when visualised, let's see the change side by side before and after capping

In [ ]:
urls = []
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/After%20capping/beforeAfterKDE.png?raw=true")
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/After%20capping/beforeAfterBoxPlot.png?raw=true")
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/After%20capping/beforeAfterCM.png?raw=true")
urls.append("https://github.com/mzst-26/water-quality-ml/blob/main/Analytics/After%20capping/beforeAfterPairPlot.png?raw=true")

for url in urls:
    display(Image(url=url))

## Summary of the most important changes we can see
>The outliers were not just noise; they were suppressing the true correlation between other nutrients and NITRITE. Before, hundreds of black dots were above the NITRITE whisker. After, it became completely clean, no dots at all. The Correlation Heatmap shows improvements between the relationship of NITRITE and other nutrients, such as NITRATE, SILICATE and PHOSPHATE. The most dramatic change being the NITRATE vs PHOSPHATE, which doubled from 0.15 to 0.30.

>NITRITE correlations with other variables strengthened significantly after capping, bimodal distributions emerged in NITRITE and AMMONIA, and scatter plot relationships became more visible. This confirms that the outliers were distorting the true signal in the data rather than representing meaningful variation, and that the cleaned dataset will give models a far clearer signal to learn from.

### Data Preparation: 4 Encode DEPTH as Categorical

In [ ]:
# Encode the 'Depth' column using one-hot encoding if it exists and hasn't been encoded yet
# This is done on the memory level, not on disk, so the original data file remains the same.
if 'Depth' in data_file.columns:
    # Created one-hot encoded columns for each unique value in the 'Depth' column, with a prefix of 'Depth' for the new column names
    depth_dummies = pd.get_dummies(data_file['Depth'], prefix='Depth', dtype=int) # dtype=int to get 0/1 instead of True/False
    # Concatenate the original dataframe with the new one-hot encoded columns
    data_file = pd.concat([data_file, depth_dummies], axis=1)
    # Drop the original 'Depth' column as it is now encoded into the new columns
    data_file.drop(columns=['Depth'], inplace=True)
    print("Depth column found and encoded successfully.")
    
elif any(col.startswith('Depth_') for col in data_file.columns):
    print("Depth encoding already exists (Depth_ columns found).")
else:
    print("No Depth column and no Depth_ columns found.")

print(data_file.columns)
# Top 5 rows of the final prepared data file
data_file.head()

#### Why does this step matter?
>Previously, we had Debt as a number (0, 10, 20, 30, 40, 60). The model would read the ddepthas continues and assumes that depth 60 is 6 times more important than 10. But they are only locations and not a factor of whether it's important. That is why converting them to dummy columns tells the model there are 6 categories of depth with no relationship between them.

#### When would I not do this?
>If Depth has a continuous relationship with Nitrite, meaning the deeper it goes, the more/less Nitrite we'd find, then we would keep it as numeric. But the pairwise plots showed that depth colour never separated cleanly from other variables, which indicates the relationship is not smooth or continuous.



### Data Preparation: 5  Check Multicollinearity

In [ ]:
# create a copy of the data file without the target variable 'NITRITE' to check for multicollinearity among the features
temp_copy = data_file.drop(columns=['NITRITE'])
# Create an empty DataFrame to store VIF values
vif_data = pd.DataFrame()
# Fill the first column of the table with the feature names
vif_data['Feature'] = temp_copy.columns
# for each feature calculate the VIF score 
# What it actually does internally is run a regression predicting that feature from all other features and measure how well it can be predicted. 
# High predictability = high VIF = redundant feature.
vif_data['VIF'] = [variance_inflation_factor(temp_copy.values, i) 
                   for i in range(temp_copy.shape[1])]

#Prints the results sorted from highest VIF to lowest so the most problematic features appear first.
print(vif_data.sort_values('VIF', ascending=False))

#### VIF GUID:
- Below 5: low multicollinearity, feature is contributing independently
- 5 to 10: moderate, worth monitoring
- Above 10: severe, feature is largely redundant
#### VIF Results Evaluation
>VIF analisis confirmed that all the features contribute independently with a score of blow 5 meanning no problimatic multicollinearity, despite the correlation observed between the NITRATE+NITRITE and PHOSPHRATE (0.85). Each feature contributes sufficiently unique information. No features were removed as a result of this analysis.




### Data Preparation: 6 Split Before Scaling


In [ ]:
# Copy dataset without NITRITE for input 
X = data_file.drop(columns=['NITRITE'])

# Target variable, which is only the NITRITE column
y = data_file['NITRITE']

# Create four separate variables for the training and testing sets using an 80-20 split of the data.
# X_train is the feature rows the model trains from
# X_test is the feature rows the model is tested on after training
# y_train is the correct nitrite answers for the training rows
# y_test is the correct nitrite answers for the test rows, used to check predictions against
# Before training, the function randomly shuffles all rows so the split is not biased towards any particular order(random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # Reserves 20% of rows for testing and uses 80% for training.
    random_state=42     # ensures same split on every run
)

# confirm the split worked by outputting the number of rows in the training and testing sets
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

#### Evaluation of the split
>The slit was successful, as the output confirms there are 1996 rows for training and 500 testing rows. 